### Step 1: Extract
โหลดข้อมูลเบื้องต้นจากไฟล์ CSV

In [16]:
import sqlite3
import pandas as pd

# อ่านไฟล์ CSV
df = pd.read_csv("retail_logs.csv")

# หาคอลัมน์ที่เป็นวันที่และแปลง format ให้ถูกต้อง
date_col = [col for col in df.columns if "date" in col.lower() or "time" in col.lower()][0]
df[date_col] = pd.to_datetime(df[date_col], format='mixed')

print(f"โหลดข้อมูลสำเร็จ: {len(df)} แถว")
display(df.head())

โหลดข้อมูลสำเร็จ: 325 แถว


,Sale_ID,Store_Code,Branch,Province,Region,Product_Name,Category,Sale_Date,Quantity,Unit_Price,Discount_Percent
0,SALE-00264,KKN-01,Khon Kaen Center,Khon Kaen,Northeast,Cookie Box,Bakery,2026-03-15,4,150.0,10.0
1,SALE-00077,PKT-01,Phuket Town,Phuket,NaN,TRAVEL MUG,Merchandise,2026-05-14,4,320.0,5.0
2,SALE-00221,CBI-01,Bangsaen,Chonburi,East,Tote Bag,Merchandise,2026-05-06,6,180.0,15.0
3,SALE-00150,PKT-01,Phuket Town,Phuket,South,caesar salad,Food,2026-03-26,2,110.0,0.0
4,SALE-00084,PKT-01,Phuket Town,PHUKET,South,Mineral Water,Beverage,2026-05-01,6,25.0,10.0


### Step 2: Transform
สร้าง Dimension Tables (Date, Location, Product) และ Fact Table

In [17]:
# 2.1 Dimension: Date
df_date = pd.DataFrame({date_col: df[date_col].drop_duplicates()}).reset_index(drop=True)
dim_date = pd.DataFrame({
    "date_id": df_date[date_col].dt.strftime("%Y%m%d").astype(int),
    "full_date": df_date[date_col].dt.date,
    "year": df_date[date_col].dt.year,
    "quarter": df_date[date_col].dt.quarter,
    "month": df_date[date_col].dt.month,
    "day": df_date[date_col].dt.day,
    "day_of_week": df_date[date_col].dt.day_name()
}).drop_duplicates()

# 2.2 Dimension: Location
location_cols = [c for c in df.columns if any(k in c.lower() for k in ["store", "branch", "location"])]
dim_location = df[location_cols].drop_duplicates().reset_index(drop=True)
dim_location.insert(0, "location_id", dim_location.index + 1)

# 2.3 Dimension: Product
product_cols = [c for c in df.columns if any(k in c.lower() for k in ["product", "item", "sku"])]
dim_product = df[product_cols].drop_duplicates().reset_index(drop=True)
dim_product.insert(0, "product_id", dim_product.index + 1)

# 2.4 Fact Table: Sales
fact_sales = df.copy()
fact_sales["date_id"] = fact_sales[date_col].dt.strftime("%Y%m%d").astype(int)

# Merge IDs กลับเข้า Fact Table
fact_sales = fact_sales.merge(dim_location, on=location_cols)
fact_sales = fact_sales.merge(dim_product, on=product_cols)

# เลือกคอลัมน์ที่เป็นตัวเลข (Measures)
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
final_cols = ["date_id", "location_id", "product_id"] + [c for c in numeric_cols if "id" not in c.lower()]
fact_sales = fact_sales[final_cols].drop_duplicates()

print("การทำ Data Transformation เสร็จสิ้น")
display(fact_sales.head())

การทำ Data Transformation เสร็จสิ้น


,date_id,location_id,product_id,Quantity,Unit_Price,Discount_Percent
0,20260315,1,1,4,150.0,10.0
1,20260514,2,2,4,320.0,5.0
2,20260506,3,3,6,180.0,15.0
3,20260326,2,4,2,110.0,0.0
4,20260501,2,5,6,25.0,10.0


### Step 3: Load
นำ Data Frames เข้าสู่ SQLite Database (`retail_warehouse.db`)

In [18]:
conn = sqlite3.connect("retail_warehouse.db")

dim_date.to_sql("dim_date", conn, if_exists="replace", index=False)
dim_location.to_sql("dim_location", conn, if_exists="replace", index=False)
dim_product.to_sql("dim_product", conn, if_exists="replace", index=False)
fact_sales.to_sql("fact_sales", conn, if_exists="replace", index=False)

conn.close()
print("Loading Data เข้า Database สำเร็จ!")

Loading Data เข้า Database สำเร็จ!


### Step 4: Verify Results
ทดสอบ Query ข้อมูลจาก Database ที่สร้างขึ้น

In [19]:
conn = sqlite3.connect("retail_warehouse.db")
query = """
SELECT d.full_date, l.Branch, p.Product_Name, f.Quantity
FROM fact_sales f
JOIN dim_date d ON f.date_id = d.date_id
JOIN dim_location l ON f.location_id = l.location_id
JOIN dim_product p ON f.product_id = p.product_id
LIMIT 10
"""
df_check = pd.read_sql(query, conn)
conn.close()
display(df_check)

,full_date,Branch,Product_Name,Quantity
0,2026-03-15,Khon Kaen Center,Cookie Box,4
1,2026-05-14,Phuket Town,TRAVEL MUG,4
2,2026-05-06,Bangsaen,Tote Bag,6
3,2026-03-26,Phuket Town,caesar salad,2
4,2026-05-01,Phuket Town,Mineral Water,6
5,2026-05-04,Siam Square,Chocolate Cake,6
6,2026-03-13,nimman,Travel Mug,2
7,2026-04-29,Phuket Town,cookie box,8
8,2026-06-28,Siam Square,Thai Milk Tea,1
9,2026-05-17,Khon Kaen Center,Tote Bag,4
